RFS 要预测的是无复发生存时间，所以这次先看有效标签有多少条、数值范围多大

In [1]:
import pandas as pd

# Load the original training spreadsheet for the RFS task.
# 重新读取原始训练表，让有有效 RFS 标签的患者都能参与这项任务。
rfs_data = pd.read_excel("TrainDataset2026.xls")

# Convert the RFS missing-value marker to NaN.
# 把 RFS 标签中的缺失标记 999 转成 NaN。
rfs_target = rfs_data["RelapseFreeSurvival (outcome)"].replace(
    [999, "999"], float("nan")
)

# Count valid and missing labels, then inspect the target distribution.
# 统计有效和缺失标签的数量，再查看目标值的分布。
print("Valid RFS labels:", rfs_target.notna().sum())
print("Missing RFS labels:", rfs_target.isna().sum())
print(rfs_target.describe())

Valid RFS labels: 400
Missing RFS labels: 0
count    400.000000
mean      56.000208
std       27.137584
min        0.000000
25%       38.000000
50%       55.000000
75%       73.000000
max      144.000000
Name: RelapseFreeSurvival (outcome), dtype: float64


RFS 有 400 条有效标签，可以全部参与划分。标签中位数是 55，范围是 0–144。作业说明用 999 标记缺失，所以目前先保留 0。

下一步做一个中位数基线：对每位验证集患者都预测同一个值，也就是训练集的 RFS 中位数。算出的 **MAE（平均绝对误差）**用来判断后续模型有没有带来提升。

In [2]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Remove the patient ID and both outcome columns from the input features.
# 从输入特征中去掉患者 ID 和两个结局列。
rfs_features = rfs_data.drop(
    columns=["ID", "pCR (outcome)", "RelapseFreeSurvival (outcome)"]
).replace([999, "999"], float("nan"))

# Hold out 20% of the patients for validation and reuse this split for later models.
# 留出 20% 的患者用于验证，后续回归模型沿用这次划分。
X_rfs_train, X_rfs_val, y_rfs_train, y_rfs_val = train_test_split(
    rfs_features, rfs_target, test_size=0.20, random_state=42
)

# Learn the median RFS from the training labels and predict it for every patient.
# 只用训练集标签计算 RFS 中位数，并把它作为每位验证集患者的预测值。
rfs_baseline = DummyRegressor(strategy="median")
rfs_baseline.fit(X_rfs_train, y_rfs_train)
rfs_baseline_predictions = rfs_baseline.predict(X_rfs_val)

# Calculate the mean absolute error on the held-out patients.
# 在留出的验证集上计算平均绝对误差。
rfs_baseline_mae = mean_absolute_error(y_rfs_val, rfs_baseline_predictions)
print(f"Training median RFS: {y_rfs_train.median():.2f}")
print(f"Validation MAE: {rfs_baseline_mae:.3f}")

Training median RFS: 55.00
Validation MAE: 21.670


RFS 的中位数基线跑通了。对每位验证集患者都预测 55，得到的 MAE 是 21.670。下一步用同一批 80 名验证集患者测试岭回归（Ridge Regression），看看加入临床和 MRI 特征后，误差能不能降下来

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Fill missing values in these clinical categories with their training-set modes.
# 这些临床类别特征的缺失值用各列在训练集中的众数填补。
rfs_mode_columns = [
    "ER", "HER2", "PgR", "TrippleNegative", "ChemoGrade",
    "HistologyType", "LNStatus", "Proliferation",
]

# Keep missing Gene values as a separate category, just as in the pCR model.
# 和 pCR 模型一样，把 Gene 的缺失值作为独立类别。
gene_pipeline = make_pipeline(
    SimpleImputer(strategy="constant", fill_value=-1),
    OneHotEncoder(categories=[[-1, 0, 1]], handle_unknown="ignore"),
)

# Use the training-set median for missing values in the remaining numeric features.
# 其余数值特征有缺失时，用各列在训练集中的中位数填补。
rfs_preprocessing = ColumnTransformer(
    [
        ("gene", gene_pipeline, ["Gene"]),
        ("clinical_mode", SimpleImputer(strategy="most_frequent"), rfs_mode_columns),
    ],
    remainder=SimpleImputer(strategy="median"),
    sparse_threshold=0,
)

# Scale the features and fit a ridge regression model with a starting alpha of 1.0.
# 标准化特征，再训练岭回归。这一版先把正则化强度 alpha 设为 1.0。
rfs_ridge = make_pipeline(rfs_preprocessing, StandardScaler(), Ridge(alpha=1.0))
rfs_ridge.fit(X_rfs_train, y_rfs_train)

# Compare both models on the same validation patients. Lower MAE is better.
# 在同一批验证患者上比较两个模型，MAE 越低越好。
rfs_ridge_predictions = rfs_ridge.predict(X_rfs_val)
rfs_ridge_mae = mean_absolute_error(y_rfs_val, rfs_ridge_predictions)

print(f"Median baseline MAE: {rfs_baseline_mae:.3f}")
print(f"Ridge validation MAE: {rfs_ridge_mae:.3f}")

Median baseline MAE: 21.670
Ridge validation MAE: 22.657


RFS 的中位数基线跑通了。对每位验证集患者都预测 55，得到的 MAE 是 21.670。下一步用同一批 80 名验证集患者测试岭回归（Ridge Regression），看看加入临床和 MRI 特征后，误差能不能降下来。

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Fill missing values in these clinical categories with their training-set modes.
# 这些临床类别特征的缺失值用各列在训练集中的众数填补。
rfs_mode_columns = [
    "ER", "HER2", "PgR", "TrippleNegative", "ChemoGrade",
    "HistologyType", "LNStatus", "Proliferation",
]

# Keep missing Gene values as a separate category, just as in the pCR model.
# 和 pCR 模型一样，把 Gene 的缺失值作为独立类别。
gene_pipeline = make_pipeline(
    SimpleImputer(strategy="constant", fill_value=-1),
    OneHotEncoder(categories=[[-1, 0, 1]], handle_unknown="ignore"),
)

# Use the training-set median for missing values in the remaining numeric features.
# 其余数值特征有缺失时，用各列在训练集中的中位数填补。
rfs_preprocessing = ColumnTransformer(
    [
        ("gene", gene_pipeline, ["Gene"]),
        ("clinical_mode", SimpleImputer(strategy="most_frequent"), rfs_mode_columns),
    ],
    remainder=SimpleImputer(strategy="median"),
    sparse_threshold=0,
)

# Scale the features and fit a ridge regression model with a starting alpha of 1.0.
# 标准化特征，再训练岭回归。这一版先把正则化强度 alpha 设为 1.0。
rfs_ridge = make_pipeline(rfs_preprocessing, StandardScaler(), Ridge(alpha=1.0))
rfs_ridge.fit(X_rfs_train, y_rfs_train)

# Compare both models on the same validation patients. Lower MAE is better.
# 在同一批验证患者上比较两个模型，MAE 越低越好。
rfs_ridge_predictions = rfs_ridge.predict(X_rfs_val)
rfs_ridge_mae = mean_absolute_error(y_rfs_val, rfs_ridge_predictions)

print(f"Median baseline MAE: {rfs_baseline_mae:.3f}")
print(f"Ridge validation MAE: {rfs_ridge_mae:.3f}")

Median baseline MAE: 21.670
Ridge validation MAE: 22.657


这次验证里，岭回归的 MAE 是 22.657，比中位数基线的 21.670 高了 0.987。我们目前只试了 alpha=1。下一步在 320 条训练记录上做 5 折交叉验证，看看增大 alpha、加强对模型系数的约束后，误差会不会下降。两种模型用相同的折来比较，这一步先留着那 80 条验证记录。

In [5]:
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_val_score

# Use identical folds for the baseline and all ridge settings.
# 基线和各组岭回归参数共用相同的五折划分。
rfs_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Negate scikit-learn's scores to report MAE on its usual positive scale.
# 将评分取反，换回通常报告的正数 MAE，数值越小越好。
baseline_fold_mae = -cross_val_score(
    rfs_baseline, X_rfs_train, y_rfs_train,
    cv=rfs_cv, scoring="neg_mean_absolute_error",
)
print(f"Median baseline mean CV MAE: {baseline_fold_mae.mean():.3f}")

# Compare the original alpha with three stronger regularisation settings.
# 对比原来的 alpha 和三个更强的正则化设置。
for alpha in [1, 10, 100, 1000]:
    model = clone(rfs_ridge).set_params(ridge__alpha=alpha)

    # Refit the full pipeline inside each fold, including imputation and scaling.
    # 每一折都重新拟合整条流水线，包括缺失值填补和标准化。
    fold_mae = -cross_val_score(
        model, X_rfs_train, y_rfs_train,
        cv=rfs_cv, scoring="neg_mean_absolute_error",
    )
    print(f"Ridge alpha={alpha:4d} | mean CV MAE: {fold_mae.mean():.3f}")

Median baseline mean CV MAE: 21.708
Ridge alpha=   1 | mean CV MAE: 34.582
Ridge alpha=  10 | mean CV MAE: 26.497
Ridge alpha= 100 | mean CV MAE: 21.594
Ridge alpha=1000 | mean CV MAE: 20.954


alpha=1000 是这四组里交叉验证表现最好的，平均 MAE 为 20.954，比同样五折里的中位数基线 21.708 低了 0.754。下一步把它放到之前留出的 80 条验证记录上，看看误差是否也比验证集基线 21.670 低。

In [6]:
# Refit the selected ridge setting on all 320 training patients.
# 用选出的 alpha=1000，在全部 320 条训练记录上重新拟合整条流水线。
rfs_candidate = clone(rfs_ridge).set_params(ridge__alpha=1000)
rfs_candidate.fit(X_rfs_train, y_rfs_train)

# Evaluate the candidate on the same 80 patients as the median baseline.
# 在与中位数基线相同的 80 条验证记录上评估候选模型。
rfs_candidate_predictions = rfs_candidate.predict(X_rfs_val)
rfs_candidate_mae = mean_absolute_error(y_rfs_val, rfs_candidate_predictions)

# A positive reduction means the candidate has lower validation error.
# 误差降低量为正，表示候选模型的验证误差更小。
print(f"Median baseline validation MAE: {rfs_baseline_mae:.3f}")
print(f"Ridge alpha=1000 validation MAE: {rfs_candidate_mae:.3f}")
print(f"MAE reduction: {rfs_baseline_mae - rfs_candidate_mae:.3f}")

Median baseline validation MAE: 21.670
Ridge alpha=1000 validation MAE: 21.980
MAE reduction: -0.310


alpha=1000 的岭回归在交叉验证里优于基线，但在留出的 80 条验证记录上，MAE 是 21.980，比基线的 21.670 高了 0.310。先保留这两组结果。

下一步做一个特征对照实验。用同样的岭回归参数和五折划分，对比「全部特征」和「只用目前代码里已确认的 10 个临床特征」。这样可以初步看看，加入 MRI 特征后，RFS 的预测误差是降低了还是升高了。先在 320 条训练记录上做这个比较。

In [7]:
# Use the ten clinical features already identified, including ER, HER2, and Gene.
# 使用已确认的 10 个临床特征，其中包括 ER、HER2 和 Gene。
rfs_clinical_columns = ["Age", "Gene"] + rfs_mode_columns
X_rfs_clinical = X_rfs_train[rfs_clinical_columns]

# Reuse the same ridge settings so only the input feature set changes.
# 沿用相同的岭回归设置，这次只改变输入特征。
rfs_clinical_model = clone(rfs_ridge).set_params(ridge__alpha=1000)

# Evaluate with the same five folds used for the full-feature model.
# 使用全特征模型之前用过的同一套五折划分。
clinical_fold_mae = -cross_val_score(
    rfs_clinical_model,
    X_rfs_clinical,
    y_rfs_train,
    cv=rfs_cv,
    scoring="neg_mean_absolute_error",
)

# Report the fold scores and their mean. Lower MAE is better.
# 输出每折的 MAE 和均值，数值越低越好。
print("Clinical-only fold MAE:", clinical_fold_mae.round(3))
print(f"Clinical-only mean CV MAE: {clinical_fold_mae.mean():.3f}")

Clinical-only fold MAE: [24.139 20.517 17.989 23.066 19.845]
Clinical-only mean CV MAE: 21.111


RFS 的五折结果齐了。全特征岭回归的平均 MAE 是 20.954，10 个临床特征是 21.111，中位数基线是 21.708。 全特征模型在交叉验证里领先，但它在 80 条验证记录上仍比基线差 0.310。所以先用全特征岭回归作为候选模型，把测试样例和 CSV 导出流程跑通。

补充一点：临床子集用了 10 个特征，而作业说明列了 11 个。写报告时可以说「全特征模型略优于 10 个临床特征的模型」，这次实验还无法单独量化 MRI 的贡献。

In [8]:
# Refit the candidate pipeline on all 400 patients with known RFS labels.
# 用全部 400 条有有效 RFS 标签的记录重新训练候选模型。
rfs_sample_model = clone(rfs_ridge).set_params(ridge__alpha=1000)
rfs_sample_model.fit(rfs_features, rfs_target)

# Read the example file and select the same features in the same order.
# 读取测试样例，按训练时的列名和顺序选取特征。
rfs_sample = pd.read_excel("TestDatasetExample.xls")
rfs_sample_features = rfs_sample.loc[:, rfs_features.columns].replace(
    [999, "999"], float("nan")
)

# Export one row per patient, with the ID first and the RFS prediction second.
# 每名患者输出一行，第一列是患者 ID，第二列是 RFS 预测值。
rfs_sample_predictions = pd.DataFrame({
    "ID": rfs_sample["ID"],
    "RFS": rfs_sample_model.predict(rfs_sample_features),
})
rfs_sample_predictions.to_csv("RFSPrediction_example.csv", index=False)
display(rfs_sample_predictions)

,ID,RFS
0,TRG002728,53.260571
1,TRG002649,52.442652
2,TRG002628,47.200299
